# QUIBC — Ablation Study

Validates the contribution of each architectural component:
- Cayley unitary transformations
- STE binarization
- Adaptive λ (rate–distortion multiplier)

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from quibc import (
    QUIBC, QUIBCModelFixedLambda,
    make_encoder, make_decoder,
    make_encoder_no_unitary, make_encoder_no_ste,
)
from quibc.train import build_clic_datasets, exponential_decay_schedule
IMG_SIZE, LATENT_CH, BATCH, EPOCHS, SEED = 256, 96, 16, 38, 42
tf.keras.utils.set_random_seed(SEED)

## 1. Dataset

In [ ]:
train_ds, val_ds = build_clic_datasets(img_size=IMG_SIZE, batch_size=BATCH, seed=SEED, cache=True)

## 2. Train All Variants

In [ ]:
def build_and_train(encoder_fn, model_cls=QUIBC, lam=1e-3, lr=1e-4, name='model'):
    enc = encoder_fn(IMG_SIZE, LATENT_CH)
    dec = make_decoder(IMG_SIZE, LATENT_CH)
    m = model_cls(enc, dec, lam=lam)
    m.compile(optimizer=tf.keras.optimizers.Adam(
        exponential_decay_schedule(lr),
        clipnorm=1.0
    ))
    _ = m(tf.zeros((1,IMG_SIZE,IMG_SIZE,3)), training=False)
    print(f'\nTraining: {name}')
    hist = m.fit(train_ds, epochs=EPOCHS, validation_data=val_ds,
                 callbacks=[tf.keras.callbacks.TerminateOnNaN()], verbose=1)
    return m, hist

In [ ]:
results = {}

# Full QUIBC (baseline)
m_full, h_full = build_and_train(make_encoder, QUIBC, name='Full QUIBC')
results['Full QUIBC'] = {'model': m_full, 'history': h_full}

# No unitary transform
m_nou, h_nou = build_and_train(make_encoder_no_unitary, QUIBC, lr=5e-5, name='No Unitary')
results['No Unitary'] = {'model': m_nou, 'history': h_nou}

# No STE
m_noste, h_noste = build_and_train(make_encoder_no_ste, QUIBC, name='No STE')
results['No STE'] = {'model': m_noste, 'history': h_noste}

# Fixed lambda
m_fixlam, h_fixlam = build_and_train(make_encoder, QUIBCModelFixedLambda, name='Fixed Lambda')
results['Fixed Lambda'] = {'model': m_fixlam, 'history': h_fixlam}

## 3. Results Summary

In [ ]:
import pandas as pd
rows = []
for name, r in results.items():
    h = r['history'].history
    rows.append({
        'Variant': name,
        'Val PSNR (dB)': f"{max(h.get('val_psnr',[0])):.2f}",
        'Val MS-SSIM':   f"{max(h.get('val_ms_ssim',[0])):.4f}",
        'Val Rate (bits)': f"{min(h.get('val_rate_bits',[0])):.4f}",
    })
df = pd.DataFrame(rows).set_index('Variant')
print(df.to_string())

## 4. Visual Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics_to_plot = [('val_psnr','Val PSNR (dB)'),('val_ms_ssim','Val MS-SSIM'),('val_rate_bits','Val Rate (bits)')]
for ax, (key, label) in zip(axes, metrics_to_plot):
    for name, r in results.items():
        vals = r['history'].history.get(key, [])
        if vals:
            ax.plot(vals, label=name)
    ax.set_xlabel('Epoch'); ax.set_ylabel(label); ax.set_title(label)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.suptitle('QUIBC Ablation Study', fontsize=14)
plt.tight_layout(); plt.show()

## 5. Bar Chart: Final Performance

In [ ]:
names = list(results.keys())
psnrs    = [max(r['history'].history.get('val_psnr',[0]))   for r in results.values()]
ms_ssims = [max(r['history'].history.get('val_ms_ssim',[0])) for r in results.values()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#2196F3','#FF5722','#4CAF50','#9C27B0']
ax1.bar(names, psnrs, color=colors, alpha=0.85)
for i,v in enumerate(psnrs): ax1.text(i, v+0.05, f'{v:.2f}', ha='center', fontweight='bold')
ax1.set_ylabel('PSNR (dB)'); ax1.set_title('Peak PSNR per Variant'); ax1.grid(True, alpha=0.3, axis='y')
ax2.bar(names, ms_ssims, color=colors, alpha=0.85)
for i,v in enumerate(ms_ssims): ax2.text(i, v+0.001, f'{v:.4f}', ha='center', fontweight='bold')
ax2.set_ylabel('MS-SSIM'); ax2.set_title('Peak MS-SSIM per Variant'); ax2.grid(True, alpha=0.3, axis='y')
plt.suptitle('Ablation: Component Contributions', fontsize=14)
plt.tight_layout(); plt.show()